# PRO3342 — Modelagem e Simulação de Sistemas de Produção
## Projeto BVE — Modelo Conceitual
### Consultoria G009 — Grupo G009

| Integrante | Número USP |
|:--|--:|
| Carla Sahuquillo | 18401953 |
| Chloé Brangerieau | 18420440 |
| Guilherme Teixeira Costa | 13725570 |
| José Pedro Coimbra | 15638527 |

**Versão:** 0.3 | **Data:** 23/09/2026

| Versão | Data | Alterações |
|:--|:--|:--|
| 0.1 | 23/09/2026 | Estrutura preliminar dos quatro processos |
| 0.2 | 23/09/2026 | Identificação do G009 e integrantes |
| 0.3 | 23/09/2026 | Adequação ao modelo da disciplina; eventos, estados, parâmetros e regras explícitas |

Este documento operacionaliza o objetivo, o escopo e os indicadores da proposta G009. A descrição da seção 3 de Costa e Mesquita (2023) é referência; particularidades da BVE permanecem sujeitas à confirmação.



## 1. Elementos do sistema

| Elemento | Tipo | Atributos | Quantidade / capacidade |
|:--|:--|:--|:--|
| Navio | Entidade externa | ID, classe, ETA, chegada, atracação, saída, terminal e disponibilidade | Conforme histórico; uma posição informada por instante |
| Pedido | Entidade | ID, navio, produto, quantidade, confirmação, liberação, prazo comercial, janela física, prioridade, estado | Uma barcaça por pedido; sem fracionamento no caso-base |
| Barcaça | Recurso móvel | ID, capacidade por produto, estoque, reservas, vazão, posição, calendário e estado | A confirmar; referência do artigo: 6, de 1.500 a 3.000 t |
| Base BVE | Estoque e recursos | Berços, bombas, compatibilidades, oferta, calendário, fila e vazão | A confirmar; artigo: 2 berços; bomba compartilhada não será duplicada |
| Terminais do porto | Localizações externas | ID, posição, restrições e janelas dos navios | Cadastro de pontos; atracação exógena |
| Canal | Rede de deslocamento | Origem/destino, distância, restrições, tempos por barcaça | Sem fila própria inicialmente; efeitos incorporados aos tempos observados |
| Fundeio | Localização de espera | ID, acesso e permissões de permanência | Capacidade não limitante como hipótese; sem abastecimento no caso-base |
| PCO | Controlador lógico | Pedidos conhecidos, prioridades, reservas, programação e horizonte | Sem fila administrativa própria inicialmente |



## 2. Processos

| Processo | Sequência | Encontro e espera |
|:--|:--|:--|
| Demanda | Comunicado → pedido → chegada → espera externa → atracação/operação portuária → disponibilidade para bunker → atendimento → desatracação/saída | Navio aguarda barcaça dentro da janela; operação portuária fornece restrições externas |
| Abastecimento | Despacho → canal → chegada → atracação lateral/preparação → bombeamento → desatracação → próximo destino | Barcaça aguarda liberação do navio; pedido e combustível ficam reservados |
| Carregamento | Retorno → fila → alocação de berço e bomba → preparação → carga → liberação | Barcaça aguarda recursos e estoque; alocação conjunta evita bloqueio parcial |
| Coordenação | Recebimento → análise de aceitação → priorização → reserva → despacho → atualização | Pedido sem solução factível aguarda revisão; janela vencida gera perda registrada |

O bombeamento dura quantidade/vazão efetiva do par barcaça–navio. Manobras, preparação e liberação são somadas separadamente, sem duplicar tempos já incluídos nos registros. Uma barcaça atende um navio por vez; estoques são controlados por produto.

Na recarga, solicita-se a quantidade da próxima missão menos o saldo disponível, limitada pela capacidade livre. O PCO não programa missão acima da capacidade compatível. A fila do terminal usa FIFO entre barcaças elegíveis; sem estoque, a carga não inicia. A política de carga por missão, em vez de carga completa, é hipótese a confirmar.



## 3. Regras de decisão do PCO

1. Ao receber solicitação, verificar produto, local, volume, janela e existência de uma programação factível, incluindo recarga. Aceitar e reservar somente quando factível; caso contrário, registrar recusa/pendência comercial. No replay histórico, reproduzir decisões observadas.
2. No teste de capacidade, manter todos os pedidos compatíveis oferecidos no denominador, inclusive recusados por falta de capacidade, evitando melhorar artificialmente o serviço pela rejeição de pedidos.
3. Reprogramar às 9h30 e 16h30, com horizonte de 36 h, e após mudanças de janela, cancelamentos, falhas ou reparos. Frequência e horizonte vêm do artigo e exigem confirmação. Novos recursos livres acionam despacho dos pedidos já programados.
4. Ordenar: passageiros; pedidos com prazo comercial no dia; menor fim de janela; confirmação mais antiga; ID. Para cada pedido, escolher barcaça compatível com estoque livre e conclusão factível, minimizando deslocamento; desempatar por término previsto e ID.
5. Sem barcaça carregada, buscar plano com recarga e chegada factível. Se não houver, manter pendência e reavaliar no próximo evento; não alocar o mesmo estoque ou pedido duas vezes.
6. Após atendimento, seguir ao próximo pedido factível com estoque; se precisar recarregar, retornar à base. Sem missão, aguardar no fundeio autorizado e desocupar o ponto do navio. Se isso não for permitido, adotar local indicado pela BVE, incluindo deslocamento.
7. Operações iniciadas ficam fixas, salvo impedimento físico; reservas futuras podem ser refeitas. Cancelamento libera reservas e reavalia destino da barcaça.
8. Janela em risco aciona reprogramação. Não iniciar serviço cujo término previsto exceda a saída física. Se uma perturbação impedir conclusão, registrar quantidade parcial e falha de serviço; eventual extensão da permanência requer regra explícita da BVE.

A alternativa de menor folga ordena pelo fim da janela menos o término previsto, mantendo compatibilidades e restrições. Prazo comercial e saída física são distintos: atraso comercial pode existir sem violar a janela física.



## 4. Eventos e variáveis de estado

| Evento | Disparado por | Efeito no estado |
|:--|:--|:--|
| Comunicado/pedido | Histórico ou gerador | Insere demanda conhecida; avalia aceitação |
| Chegada/atracação/liberação | Cronograma externo | Atualiza localização, janela e elegibilidade |
| Alteração/cancelamento | Agência/comercial | Atualiza pedido, desfaz reservas futuras e reprograma |
| Despacho/chegada da barcaça | PCO/fim da viagem | Reserva missão; muda posição e estado para viagem/espera |
| Início/fim do abastecimento | Recursos prontos/fim da duração | Ocupa/libera barcaça; contabiliza entrega e saldo |
| Chegada à base/início de carga | Fim da viagem/recursos elegíveis | Insere/remove fila; ocupa berço e bomba conjuntamente |
| Fim de carga | Duração amostrada/calculada | Atualiza estoques; libera recursos |
| Reposição | Agenda de suprimento | Acrescenta combustível à base respeitando capacidade |
| Falha/reparo/bloqueio | Histórico ou gerador | Altera disponibilidade, interrompe quando aplicável e reprograma |
| Revisão do PCO | Horário ou exceção | Recalcula alocações futuras |
| Saída/fim de janela | Agenda externa | Encerra elegibilidade; registra pedidos incompletos |

O estado guarda posição, estoque e reservas por barcaça; ocupação de berços/bombas; estoque da base; filas; programação; e situação de cada pedido. Para eventos simultâneos: contabilizar conclusões/reposições, atualizar restrições e saídas e então executar uma rodada de despacho. Conclusão exatamente no fim da janela é pontual.



## 5. Parâmetros e variáveis aleatórias

| Parâmetro / variável | Símbolo | Unidade | Natureza |
|:--|:--|:--|:--|
| Frota, berços, bombas | M, B, P | unidades | Constantes por cenário; cadastro |
| Capacidade útil por produto | Kkp | t | Constante por recurso |
| Estoques/posições iniciais | Skp(0), Xk(0) | t; local | Condições observadas |
| Intervalos entre pedidos, quantidades, janelas | A, Q, W | h; t; data/hora | Aleatórias conjuntas; preservar sazonalidade e dependências |
| Deslocamento/manobra/preparação | Tviagem, Tprep | h | Aleatórias condicionais a rota, recurso e período |
| Vazão efetiva | F | t/h | Estimada por quantidade e duração; capacidade nominal é limite |
| Falhas, reparos e restrições climáticas | Tfalha, Treparo | h | Aleatórias; conservar eventos comuns a vários recursos |
| Reposição da base | R(t) | t; data/hora | Agenda observada ou processo estimado |
| Regra e frequência do PCO | π, Δ | categoria; h | Constantes por cenário |
| Limiar de serviço e observação | α, H | %; dias | Propostos: 95% e 90 dias após aquecimento |

Distribuições não serão escolhidas sem dados. Usar reamostragem de registros completos ou ajuste validado, evitando assumir independência entre volume, local e janela. Unidades volumétricas exigirão conversão documentada por densidade.



## 6. Indicadores

| Indicador | Cálculo a partir dos eventos | Unidade |
|:--|:--|:--|
| Volume entregue | Soma das quantidades transferidas na observação / dias | t/dia |
| Pontualidade | Pedidos integralmente concluídos na janela / pedidos compatíveis oferecidos com janela encerrada | % |
| Não atendimento | 1 − pontualidade; inclui recusas por capacidade e entregas incompletas | % |
| Atraso comercial | max(0, conclusão − prazo); informar separadamente perdas | h |
| Espera média/P95 | Início do atendimento − liberação física; apenas atendidos, sempre junto à taxa de perdas | h |
| Utilização da barcaça | Tempo em navegação, manobra, bombeamento e carga / tempo disponível | % |
| Espera/ociosidade da barcaça | Tempo em fila, espera de navio ou fundeio / tempo disponível, por estado | % |
| Utilização de berço/bomba | Tempo ocupado, incluindo preparação/liberação pertinentes / tempo disponível | % |
| Carteira | Pedidos aceitos ainda não concluídos, com t pendentes por instante | pedidos; t |
| Capacidade e ganho | Maior entrega sustentável com serviço ≥95%; (C1/C0 − 1) × 100 | t/dia; % |

Cancelamentos comerciais anteriores ao serviço serão excluídos e reportados separadamente. Demandas perdidas não desaparecem do denominador; indisponibilidades ficam fora do tempo disponível e são informadas à parte. Inicialização reproduzirá serviços em curso. Verificação examinará balanços e conflitos; validação usará período separado. Para capacidade, ampliar gradualmente a demanda preservando o perfil e usar inicialmente 20 replicações, com IC95% e ampliação conforme precisão.



## 7. Premissas e questões em aberto

Hipóteses iniciais: sem divisão de pedidos; abastecimento apenas com navio atracado; canal sem fila endógena; fundeio não limitante; ausência de gargalo administrativo. Simplificam o modelo e serão revistas se os dados indicarem relevância. Confirmar com a BVE aceitação, prioridades, política de recarga, interrupções, extensão de janela, uso do fundeio, restrições simultâneas e configurações reais. Essas questões orientam G009_Dados.
